In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, make_scorer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from xgboost import XGBClassifier
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt

random_state=42

# Load prepared data
df = pd.read_csv("../.data/modelisation/app_train.csv")
target_col = "TARGET"
X = df.drop(columns=[target_col])
y = df[target_col]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_state, stratify=y
)
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

# Instantiate model
model = XGBClassifier(random_state=random_state, eval_metric="auc")

pipeline = Pipeline([
    ('smote', SMOTE(random_state=random_state, sampling_strategy=0.5)),
    ('clf', model)
])

# Define parameter grid
param_grid = {
    'clf__n_estimators': [100, 200, 300],
    'clf__max_depth': [3, 5, 7],
    'clf__learning_rate': [0.01, 0.1, 0.3],
    'clf__subsample': [0.8, 1.0],
    'clf__colsample_bytree': [0.8, 1.0]
}

# GridSearchCV
grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=cv_splitter,
    scoring='roc_auc',
    n_jobs=2,
    verbose=3
)

# Fit GridSearch
grid_search.fit(X_train, y_train)

# Best parameters
best_params = grid_search.best_params_
best_pipeline = grid_search.best_estimator_

print(f"Meilleurs paramètres: {best_params}")
print(f"Meilleur score AUC (CV): {grid_search.best_score_:.4f}")

Fitting 5 folds for each of 108 candidates, totalling 540 fits
[CV 1/5] END clf__colsample_bytree=0.8, clf__learning_rate=0.01, clf__max_depth=3, clf__n_estimators=100, clf__subsample=0.8;, score=0.687 total time=   5.6s
[CV 4/5] END clf__colsample_bytree=0.8, clf__learning_rate=0.01, clf__max_depth=3, clf__n_estimators=100, clf__subsample=0.8;, score=0.686 total time=   5.7s


/usr/local/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV 2/5] END clf__colsample_bytree=0.8, clf__learning_rate=0.01, clf__max_depth=3, clf__n_estimators=100, clf__subsample=0.8;, score=0.684 total time=   5.5s
[CV 3/5] END clf__colsample_bytree=0.8, clf__learning_rate=0.01, clf__max_depth=3, clf__n_estimators=100, clf__subsample=0.8;, score=0.684 total time=   6.1s
[CV 1/5] END clf__colsample_bytree=0.8, clf__learning_rate=0.01, clf__max_depth=3, clf__n_estimators=100, clf__subsample=1.0;, score=0.685 total time=   6.4s
[CV 3/5] END clf__colsample_bytree=0.8, clf__learning_rate=0.01, clf__max_depth=3, clf__n_estimators=100, clf__subsample=1.0;, score=0.684 total time=   6.3s
[CV 5/5] END clf__colsample_bytree=0.8, clf__learning_rate=0.01, clf__max_depth=3, clf__n_estimators=100, clf__subsample=0.8;, score=0.685 total time=   6.1s
[CV 2/5] END clf__colsample_bytree=0.8, clf__learning_rate=0.01, clf__max_depth=3, clf__n_estimators=100, clf__subsample=1.0;, score=0.684 total time=   6.9s
[CV 4/5] END clf__colsample_bytree=0.8, clf__learnin